# ETL OpenFoodFacts - Datamart Nutrition & Qualite

**Module** : TRDE703 - Atelier Integration des Donnees (M1)  
**Technologie** : Apache Spark (PySpark) → PostgreSQL  
**Architecture** : Bronze → Silver → Gold → Datamart

---

## Sommaire
1. Configuration et imports
2. **BRONZE** - Ingestion des donnees brutes
3. **SILVER** - Nettoyage et conformite
4. **GOLD** - Modelisation dimensionnelle
5. Chargement PostgreSQL
6. Metriques de qualite

---
## 1. Configuration et Imports

### 1.1 Imports des librairies
- **PySpark** : Moteur de traitement distribue
- **psycopg2** : Connexion PostgreSQL pour TRUNCATE CASCADE
- **datetime** : Gestion des dates SCD2

In [ ]:
#CELLULE 0 -  PostgreSQL - Creation base et tables (DDL)

"""
Script de creation de la base de donnees PostgreSQL pour OpenFoodFacts
Executer ce script avant de lancer le notebook ETL
ENCODAGE UTF-8 pour compatibilite avec PySpark JDBC
"""

import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# ============================================
# CONFIGURATION
# ============================================
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"  # Ton mot de passe
DB_NAME = "openfoodfacts_dw"

# ============================================
# ETAPE 1 : Supprimer et recreer la base (UTF-8)
# ============================================
def create_database():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()

        # Verifier si la base existe
        cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
        exists = cursor.fetchone()

        if exists:
            # Fermer toutes les connexions a la base
            cursor.execute(f"""
                SELECT pg_terminate_backend(pg_stat_activity.pid)
                FROM pg_stat_activity
                WHERE pg_stat_activity.datname = '{DB_NAME}'
                AND pid <> pg_backend_pid();
            """)
            # Supprimer la base
            cursor.execute(f"DROP DATABASE {DB_NAME}")
            print(f"Base '{DB_NAME}' supprimee")

        # Creer la base en UTF-8
        cursor.execute(f"""
            CREATE DATABASE {DB_NAME}
            WITH ENCODING = 'UTF8'
            LC_COLLATE = 'French_France.1252'
            LC_CTYPE = 'French_France.1252'
            TEMPLATE = template0;
        """)
        print(f"Base '{DB_NAME}' creee (UTF-8)")

        cursor.close()
        conn.close()
        return True

    except Exception as e:
        print(f"Erreur creation base: {e}")
        # Essayer avec les locales par defaut
        try:
            conn = psycopg2.connect(
                host=DB_HOST, port=DB_PORT,
                user=DB_USER, password=DB_PASSWORD
            )
            conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
            cursor = conn.cursor()
            cursor.execute(f"CREATE DATABASE {DB_NAME} WITH ENCODING = 'UTF8' TEMPLATE = template0;")
            print(f"Base '{DB_NAME}' creee (UTF-8, locales par defaut)")
            cursor.close()
            conn.close()
            return True
        except Exception as e2:
            print(f"Erreur: {e2}")
            return False

# ============================================
# ETAPE 2 : Creer les tables
# ============================================
def create_tables():
    ddl_statements = [
        # Supprimer les tables existantes dans l'ordre inverse des FK
        "DROP TABLE IF EXISTS fact_nutrition_snapshot CASCADE;",
        "DROP TABLE IF EXISTS bridge_product_category CASCADE;",
        "DROP TABLE IF EXISTS dim_product CASCADE;",
        "DROP TABLE IF EXISTS dim_nutri CASCADE;",
        "DROP TABLE IF EXISTS dim_category CASCADE;",
        "DROP TABLE IF EXISTS dim_country CASCADE;",
        "DROP TABLE IF EXISTS dim_brand CASCADE;",
        "DROP TABLE IF EXISTS dim_time CASCADE;",
        
        # Creer les nouvelles tables
        """
        CREATE TABLE dim_time (
            time_sk BIGINT PRIMARY KEY,
            date DATE,
            year INTEGER,
            month INTEGER,
            day INTEGER,
            week INTEGER,
            iso_week INTEGER
        );
        """,
        """
        CREATE TABLE dim_brand (
            brand_sk BIGINT PRIMARY KEY,
            brand_name VARCHAR(500)
        );
        """,
        """
        CREATE TABLE dim_country (
            country_sk BIGINT PRIMARY KEY,
            country_code VARCHAR(100),
            country_name_fr VARCHAR(255)
        );
        """,
        """
        CREATE TABLE dim_category (
            category_sk BIGINT PRIMARY KEY,
            category_code VARCHAR(500),
            category_name_fr VARCHAR(500),
            level INTEGER,
            parent_category_sk BIGINT
        );
        """,
        """
        CREATE TABLE dim_nutri (
            nutri_sk BIGINT PRIMARY KEY,
            nutriscore_grade VARCHAR(1),
            nova_group INTEGER,
            ecoscore_grade VARCHAR(1)
        );
        """,
        """
        CREATE TABLE dim_product (
            product_sk BIGINT PRIMARY KEY,
            code VARCHAR(50),
            product_name VARCHAR(1000),
            brand_sk BIGINT,
            primary_category_sk BIGINT,
            countries_multi TEXT,
            effective_from DATE,
            effective_to DATE,
            is_current BOOLEAN
        );
        """,
        """
        CREATE TABLE bridge_product_category (
            product_sk BIGINT,
            category_sk BIGINT,
            PRIMARY KEY (product_sk, category_sk)
        );
        """,
        """
        CREATE TABLE fact_nutrition_snapshot (
            fact_id BIGINT PRIMARY KEY,
            product_sk BIGINT,
            time_sk BIGINT,
            energy_kcal_100g DECIMAL(10,2),
            fat_100g DECIMAL(10,2),
            saturated_fat_100g DECIMAL(10,2),
            sugars_100g DECIMAL(10,2),
            salt_100g DECIMAL(10,2),
            proteins_100g DECIMAL(10,2),
            fiber_100g DECIMAL(10,2),
            sodium_100g DECIMAL(10,2),
            nutriscore_grade VARCHAR(1),
            nova_group INTEGER,
            ecoscore_grade VARCHAR(1),
            completeness_score DECIMAL(5,4),
            quality_issues_json TEXT
        );
        """,
        # Index
        "CREATE INDEX idx_fact_product ON fact_nutrition_snapshot(product_sk);",
        "CREATE INDEX idx_fact_time ON fact_nutrition_snapshot(time_sk);",
        "CREATE INDEX idx_product_brand ON dim_product(brand_sk);",
        "CREATE INDEX idx_product_category ON dim_product(primary_category_sk);",
    ]

    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        cursor = conn.cursor()

        for ddl in ddl_statements:
            cursor.execute(ddl)

        conn.commit()
        cursor.close()
        conn.close()

        print("Tables creees avec succes")
        print("  - dim_time")
        print("  - dim_brand")
        print("  - dim_country")
        print("  - dim_category")
        print("  - dim_nutri")
        print("  - dim_product")
        print("  - bridge_product_category")
        print("  - fact_nutrition_snapshot")
        return True

    except Exception as e:
        print(f"Erreur creation tables: {e}")
        return False

# ============================================
# ETAPE 3 : Verifier
# ============================================
def verify_structure():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        cursor = conn.cursor()

        # Verifier l'encodage
        cursor.execute("SHOW server_encoding;")
        encoding = cursor.fetchone()[0]
        print(f"\nEncodage serveur: {encoding}")

        cursor.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = cursor.fetchall()

        print("\nTables creees:")
        for table in tables:
            cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
            count = cursor.fetchone()[0]
            print(f"  {table[0]}: {count} lignes")

        cursor.close()
        conn.close()
        return True

    except Exception as e:
        print(f"Erreur verification: {e}")
        return False

# ============================================
# EXECUTION
# ============================================
if __name__ == "__main__":
    print("=" * 50)
    print("Configuration PostgreSQL - OpenFoodFacts")
    print("=" * 50)
    
    if create_database():
        if create_tables():
            verify_structure()
    
    print("\n" + "=" * 50)
    print("Termine! Executez maintenant le notebook v5.4")
    print("=" * 50)

In [ ]:
# CELLULE 1 - Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType
from datetime import datetime
import psycopg2
import json

print("Imports OK")

### 1.2 Session Spark
Configuration de la session avec :
- **4 Go de memoire** pour le driver
- Mode **LEGACY** pour compatibilite des dates

In [ ]:
# CELLULE 2 - Session Spark
spark = SparkSession.builder \
    .appName("OpenFoodFacts_ETL_v6") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} initialise")

### 1.3 Configuration
- **Source** : Fichier CSV OpenFoodFacts
- **Cible** : PostgreSQL (openfoodfacts_dw)
- **Regles de qualite** : Bornes nutritionnelles, seuils d'anomalies

In [ ]:
# CELLULE 3 - Configuration
# Source
INPUT_PATH = "data/bronze/en.openfoodfacts.org.products_echantillon_3000.csv"

# PostgreSQL
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DATABASE = "openfoodfacts_dw"
PG_USER = "postgres"
PG_PASSWORD = "postgres"  # MODIFIER ICI

PG_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}?charSet=UTF-8"
PG_PROPS = {"user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}

# Regles de qualite - Bornes nutritionnelles (g/100g)
NUTRIENT_BOUNDS = {
    "energy_kcal_100g": (0, 900),
    "fat_100g": (0, 100),
    "saturated_fat_100g": (0, 100),
    "sugars_100g": (0, 100),
    "salt_100g": (0, 100),
    "proteins_100g": (0, 100),
    "fiber_100g": (0, 100),
    "sodium_100g": (0, 40)
}

# Facteur de conversion sel/sodium
SALT_SODIUM_FACTOR = 2.5

# Minimum de nutriments requis
MIN_NUTRIENTS = 3

print(f"Source: {INPUT_PATH}")
print(f"Cible: PostgreSQL {PG_DATABASE}")

### 1.4 Fonctions utilitaires
- **detect_separator()** : Detection automatique du separateur CSV
- **clean_numeric()** : Nettoyage et validation des colonnes numeriques
- **load_to_pg()** : Chargement JDBC avec TRUNCATE CASCADE

In [ ]:
# CELLULE 4 - Fonctions utilitaires

def detect_separator(path):
    """Detecte le separateur CSV (tabulation, point-virgule ou virgule)"""
    with open(path, 'r', encoding='utf-8') as f:
        line = f.readline()
    if line.count('\t') > max(line.count(';'), line.count(',')):
        return '\t'
    if line.count(';') > line.count(','):
        return ';'
    return ','

def clean_numeric(col_name):
    """Nettoie une colonne numerique : virgule->point, validation regex"""
    c = F.regexp_replace(F.col(col_name), ",", ".")
    c = F.when(c.like("http%"), None).otherwise(c)  # Exclure URLs
    c = F.when(c.rlike(r'^[+-]?[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?$'), c.cast("double")).otherwise(None)
    return c

def load_to_pg(df, table_name):
    """Charge un DataFrame dans PostgreSQL avec TRUNCATE CASCADE"""
    try:
        # TRUNCATE pour eviter les conflits de FK
        conn = psycopg2.connect(host=PG_HOST, port=int(PG_PORT), 
                                database=PG_DATABASE, user=PG_USER, password=PG_PASSWORD)
        conn.set_client_encoding('UTF8')
        cur = conn.cursor()
        cur.execute(f"TRUNCATE TABLE {table_name} CASCADE")
        conn.commit()
        cur.close()
        conn.close()
        
        # INSERT via JDBC
        df.write.jdbc(url=PG_URL, table=table_name, mode="append", properties=PG_PROPS)
        count = df.count()
        print(f"   {table_name}: {count} lignes")
        return count
    except Exception as e:
        print(f"   ERREUR {table_name}: {e}")
        return 0

print("Fonctions definies")

---
## 2. BRONZE - Ingestion des donnees brutes

**Objectif** : Charger le fichier CSV source avec extraction des champs cles.

**Champs extraits** :
- Identifiants : `code`, `product_name`
- Classifications : `brands`, `categories_tags`, `countries_tags`
- Nutriments : `energy_kcal_100g`, `fat_100g`, `sugars_100g`, etc.
- Scores : `nutriscore_grade`, `nova_group`, `ecoscore_grade`
- Metadata : `last_modified_t`, `completeness`

In [ ]:
# CELLULE 5 - Lecture Bronze
sep = detect_separator(INPUT_PATH)
print(f"Separateur detecte: '{sep}'")

df_raw = spark.read \
    .option("header", "true") \
    .option("sep", sep) \
    .option("inferSchema", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(INPUT_PATH)

bronze_count = df_raw.count()
print(f"BRONZE: {bronze_count} lignes, {len(df_raw.columns)} colonnes")

### 2.1 Selection et typage des colonnes
Extraction des colonnes necessaires avec cast explicite des types.

In [ ]:
# CELLULE 6 - Selection des colonnes avec typage explicite
df = df_raw.select(
    # Identifiants
    F.col("code").cast("string"),
    F.col("product_name").cast("string"),
    # Classifications
    F.col("brands").cast("string"),
    F.col("categories_tags").cast("string"),
    F.col("countries_tags").cast("string"),
    # Scores
    F.col("nutriscore_grade").cast("string"),
    clean_numeric("nova_group").cast("int").alias("nova_group"),
    F.col("environmental_score_grade").cast("string").alias("ecoscore_grade"),
    # Nutriments (nettoyage numerique)
    clean_numeric("energy-kcal_100g").alias("energy_kcal_100g"),
    clean_numeric("fat_100g").alias("fat_100g"),
    clean_numeric("saturated-fat_100g").alias("saturated_fat_100g"),
    clean_numeric("sugars_100g").alias("sugars_100g"),
    clean_numeric("salt_100g").alias("salt_100g"),
    clean_numeric("proteins_100g").alias("proteins_100g"),
    clean_numeric("fiber_100g").alias("fiber_100g"),
    clean_numeric("sodium_100g").alias("sodium_100g"),
    # Metadata
    clean_numeric("completeness").alias("completeness"),
    F.col("data_quality_errors_tags").cast("string").alias("quality_errors"),
    F.col("last_modified_datetime").cast("string").alias("last_modified_datetime"),
    clean_numeric("last_modified_t").cast("long").alias("last_modified_t")
)

print(f"Colonnes selectionnees: {len(df.columns)}")

---
## 3. SILVER - Nettoyage et Conformite

**Regles de qualite appliquees** :
1. Filtrage : code et product_name obligatoires
2. Deduplication : par code-barres, garder le plus recent (last_modified_t)
3. Bornes : nutriments dans intervalles valides [0-100] g/100g
4. Coherence : saturated_fat <= fat
5. Harmonisation : conversion sel/sodium (sel = 2.5 × sodium)
6. Completude : minimum 3 nutriments renseignes

In [ ]:
# CELLULE 7 - Filtrage et deduplication

# 7.1 Filtrer : code et nom obligatoires
df = df.filter(F.col("code").isNotNull() & (F.col("code") != ""))
df = df.filter(F.col("product_name").isNotNull() & (F.col("product_name") != ""))
count_after_filter = df.count()
print(f"1. Apres filtrage code/nom: {count_after_filter} (-{bronze_count - count_after_filter})")

# 7.2 Deduplication : garder le plus recent par code (last_modified_t)
window_dedup = Window.partitionBy("code").orderBy(F.col("last_modified_t").desc_nulls_last())
df = df.withColumn("_rank", F.row_number().over(window_dedup))
df = df.filter(F.col("_rank") == 1).drop("_rank")
count_after_dedup = df.count()
print(f"2. Apres deduplication (plus recent): {count_after_dedup} (-{count_after_filter - count_after_dedup})")

### 3.1 Application des regles de qualite

In [ ]:
# CELLULE 8 - Regles de qualite

# 8.1 Appliquer bornes nutritionnelles
for col_name, (min_v, max_v) in NUTRIENT_BOUNDS.items():
    if col_name in df.columns:
        df = df.withColumn(col_name, 
            F.when((F.col(col_name) >= min_v) & (F.col(col_name) <= max_v), F.col(col_name))
            .otherwise(None))
print("3. Bornes nutritionnelles appliquees")

# 8.2 Coherence : saturated_fat <= fat
df = df.withColumn("saturated_fat_100g", 
    F.when(F.col("saturated_fat_100g") > F.col("fat_100g"), None)
    .otherwise(F.col("saturated_fat_100g")))
print("4. Coherence saturated_fat <= fat")

# 8.3 Harmonisation sel/sodium
df = df.withColumn("sodium_100g", 
    F.when(F.col("sodium_100g").isNull() & F.col("salt_100g").isNotNull(),
           F.round(F.col("salt_100g") / SALT_SODIUM_FACTOR, 2))
    .otherwise(F.col("sodium_100g")))
df = df.withColumn("salt_100g", 
    F.when(F.col("salt_100g").isNull() & F.col("sodium_100g").isNotNull(),
           F.round(F.col("sodium_100g") * SALT_SODIUM_FACTOR, 2))
    .otherwise(F.col("salt_100g")))
print("5. Harmonisation sel/sodium (facteur 2.5)")

In [ ]:
# CELLULE 9 - Filtres de completude

# 9.1 Compter nutriments disponibles
nutrient_cols = ["energy_kcal_100g", "fat_100g", "saturated_fat_100g", 
                 "sugars_100g", "salt_100g", "proteins_100g", "fiber_100g"]
df = df.withColumn("_n_count", 
    sum([F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in nutrient_cols]))

# 9.2 Filtrer : minimum 3 nutriments
df = df.filter(F.col("_n_count") >= MIN_NUTRIENTS)
count_after_nutrients = df.count()
print(f"6. Apres filtre min {MIN_NUTRIENTS} nutriments: {count_after_nutrients}")

# 9.3 Somme nutriments <= 110g (coherence physique)
df = df.withColumn("_sum", 
    F.coalesce(F.col("fat_100g"), F.lit(0)) +
    F.coalesce(F.col("sugars_100g"), F.lit(0)) +
    F.coalesce(F.col("proteins_100g"), F.lit(0)) +
    F.coalesce(F.col("fiber_100g"), F.lit(0)) +
    F.coalesce(F.col("salt_100g"), F.lit(0)))
df = df.filter(F.col("_sum") <= 110)
count_after_sum = df.count()
print(f"7. Apres filtre somme <= 110g: {count_after_sum}")

### 3.2 Normalisation et enrichissement

In [ ]:
# CELLULE 10 - Normalisation

# 10.1 Normaliser scores (A-E pour nutriscore/ecoscore, 1-4 pour nova)
df = df.withColumn("nutriscore_grade", 
    F.when(F.upper(F.col("nutriscore_grade")).isin(["A","B","C","D","E"]), 
           F.upper(F.col("nutriscore_grade"))).otherwise(None))
df = df.withColumn("nova_group", 
    F.when(F.col("nova_group").between(1, 4), F.col("nova_group")).otherwise(None))
df = df.withColumn("ecoscore_grade", 
    F.when(F.upper(F.col("ecoscore_grade")).isin(["A","B","C","D","E"]), 
           F.upper(F.col("ecoscore_grade"))).otherwise(None))
print("8. Scores normalises")

# 10.2 Extraire valeurs principales
df = df.withColumn("brand_primary", F.trim(F.split(F.col("brands"), ",").getItem(0)))
df = df.withColumn("primary_category_code", F.trim(F.split(F.col("categories_tags"), ",").getItem(0)))
df = df.withColumn("primary_country_code", F.trim(F.split(F.col("countries_tags"), ",").getItem(0)))

# 10.3 Creer JSON pour champs multi-valeurs
df = df.withColumn("countries_multi",
    F.when(F.col("countries_tags").isNotNull() & (F.col("countries_tags") != ""),
           F.concat(F.lit('["'), F.regexp_replace(F.col("countries_tags"), ",", '","'), F.lit('"]')))
    .otherwise(F.lit('[]')))

df = df.withColumn("quality_issues_json",
    F.when(F.col("quality_errors").isNotNull() & (F.col("quality_errors") != ""),
           F.concat(F.lit('["'), F.regexp_replace(F.col("quality_errors"), ",", '","'), F.lit('"]')))
    .otherwise(F.lit('[]')))

print("9. Valeurs principales et JSON crees")

In [ ]:
# CELLULE 11 - Score de completude et finalisation Silver

# 11.1 Calculer completeness_score (0-1)
df = df.withColumn("completeness_score",
    F.when(F.col("completeness").isNotNull(), F.round(F.col("completeness"), 4))
    .otherwise(F.round(
        (F.when(F.col("product_name").isNotNull(), 1).otherwise(0) +
         F.when(F.col("brands").isNotNull(), 1).otherwise(0) +
         F.when(F.col("categories_tags").isNotNull(), 1).otherwise(0) +
         (F.col("_n_count") / len(nutrient_cols))) / 4.0, 4)))

# 11.2 Arrondir mesures a 2 decimales
for c in ["energy_kcal_100g", "fat_100g", "saturated_fat_100g", "sugars_100g", 
          "salt_100g", "proteins_100g", "fiber_100g", "sodium_100g"]:
    df = df.withColumn(c, F.round(F.col(c), 2))

# 11.3 Nettoyer valeurs "unknown"
for c in ["brand_primary", "primary_category_code", "primary_country_code"]:
    df = df.withColumn(c, F.when(F.lower(F.col(c)) == "unknown", None).otherwise(F.col(c)))

# 11.4 Supprimer colonnes temporaires
df_silver = df.drop("_n_count", "_sum", "completeness", "quality_errors")

silver_count = df_silver.count()
print(f"\n=== SILVER FINAL: {silver_count} produits ===")
print(f"Taux de retention: {silver_count/bronze_count*100:.1f}%")

### 3.3 Statistiques de qualite Silver

In [ ]:
# CELLULE 12 - Statistiques Silver
print("=== TAUX DE REMPLISSAGE ===")
stats_cols = ["product_name", "brand_primary", "primary_category_code", 
              "nutriscore_grade", "nova_group", "ecoscore_grade", "energy_kcal_100g"]
for c in stats_cols:
    non_null = df_silver.filter(F.col(c).isNotNull()).count()
    pct = round(non_null / silver_count * 100, 1)
    print(f"  {c:25}: {pct:5.1f}%")

avg_completeness = df_silver.agg(F.avg("completeness_score")).first()[0]
print(f"\nCompletude moyenne: {avg_completeness:.2%}")

---
## 4. GOLD - Modelisation Dimensionnelle

**Schema en etoile** :
- **dim_time** : Dimension temporelle (dates des snapshots)
- **dim_brand** : Dimension marque
- **dim_country** : Dimension pays
- **dim_category** : Dimension categorie (hierarchique)
- **dim_nutri** : Dimension scores nutritionnels (optionnel)
- **dim_product** : Dimension produit (SCD2)
- **fact_nutrition_snapshot** : Table de faits
- **bridge_product_category** : Table de pont N-N (optionnel)

### 4.1 dim_time
Dimension temporelle basee sur `last_modified_datetime`.

In [ ]:
# CELLULE 13 - dim_time
from pyspark.sql.functions import year, month, dayofmonth, weekofyear

# Extraire les dates depuis last_modified_datetime
df_dates = df_silver.withColumn("snapshot_date",
    F.when(F.col("last_modified_datetime").isNotNull(),
           F.to_date(F.col("last_modified_datetime")))
    .otherwise(F.current_date()))

df_dim_time = df_dates.select("snapshot_date").distinct() \
    .filter(F.col("snapshot_date").isNotNull()) \
    .withColumn("time_sk", F.dense_rank().over(Window.orderBy("snapshot_date"))) \
    .withColumn("year", year(F.col("snapshot_date"))) \
    .withColumn("month", month(F.col("snapshot_date"))) \
    .withColumn("day", dayofmonth(F.col("snapshot_date"))) \
    .withColumn("week", weekofyear(F.col("snapshot_date"))) \
    .withColumn("iso_week", weekofyear(F.col("snapshot_date"))) \
    .select(
        F.col("time_sk"),
        F.col("snapshot_date").alias("date"),
        "year", "month", "day", "week", "iso_week"
    )

print(f"dim_time: {df_dim_time.count()} dates distinctes")
df_dim_time.orderBy("date").show(5)

### 4.2 dim_brand

In [ ]:
# CELLULE 14 - dim_brand
df_dim_brand = df_silver \
    .select(F.explode(F.split(F.col("brands"), ",")).alias("brand_name_raw")) \
    .withColumn("brand_name", F.trim(F.col("brand_name_raw"))) \
    .filter(F.col("brand_name").isNotNull() & (F.col("brand_name") != "")) \
    .dropDuplicates(["brand_name"]) \
    .withColumn("brand_sk", F.monotonically_increasing_id()) \
    .select("brand_sk", "brand_name")

print(f"dim_brand: {df_dim_brand.count()} marques")

### 4.3 dim_country

In [ ]:
# CELLULE 15 - dim_country
df_dim_country = df_silver \
    .select(F.explode(F.split(F.col("countries_tags"), ",")).alias("country_code_raw")) \
    .withColumn("country_code", F.trim(F.col("country_code_raw"))) \
    .filter(F.col("country_code").isNotNull() & (F.col("country_code") != "")) \
    .dropDuplicates(["country_code"]) \
    .withColumn("country_sk", F.monotonically_increasing_id()) \
    .withColumn("country_name_fr", F.regexp_replace(F.col("country_code"), "^en:", "")) \
    .select("country_sk", "country_code", "country_name_fr")

print(f"dim_country: {df_dim_country.count()} pays")

### 4.4 dim_category
Dimension hierarchique avec niveau (`level`) base sur la position dans `categories_tags`.

In [ ]:
# CELLULE 16 - dim_category
df_dim_category = df_silver \
    .select(F.posexplode(F.split(F.col("categories_tags"), ",")).alias("pos", "cat_raw")) \
    .withColumn("category_code", F.trim(F.col("cat_raw"))) \
    .filter(F.col("category_code").isNotNull() & (F.col("category_code") != "")) \
    .groupBy("category_code").agg(F.min("pos").alias("level")) \
    .withColumn("level", F.col("level") + 1) \
    .withColumn("category_sk", F.monotonically_increasing_id()) \
    .withColumn("category_name_fr", F.regexp_replace(F.col("category_code"), "^en:", "")) \
    .withColumn("parent_category_sk", F.lit(None).cast("long")) \
    .select("category_sk", "category_code", "category_name_fr", "level", "parent_category_sk")

print(f"dim_category: {df_dim_category.count()} categories")

### 4.5 dim_nutri (optionnel)

In [ ]:
# CELLULE 17 - dim_nutri
df_dim_nutri = df_silver \
    .select("nutriscore_grade", "nova_group", "ecoscore_grade") \
    .dropDuplicates() \
    .filter(F.col("nutriscore_grade").isNotNull() | 
            F.col("nova_group").isNotNull() | 
            F.col("ecoscore_grade").isNotNull()) \
    .withColumn("nutri_sk", F.monotonically_increasing_id()) \
    .select("nutri_sk", "nutriscore_grade", "nova_group", "ecoscore_grade")

print(f"dim_nutri: {df_dim_nutri.count()} combinaisons")

### 4.6 dim_product (SCD2)
Dimension produit avec champs SCD2 : `effective_from`, `effective_to`, `is_current`.

In [ ]:
# CELLULE 18 - dim_product
today_str = datetime.now().strftime("%Y-%m-%d")

df_dim_product = df_silver \
    .join(df_dim_brand, df_silver["brand_primary"] == df_dim_brand["brand_name"], "left") \
    .join(df_dim_category, df_silver["primary_category_code"] == df_dim_category["category_code"], "left") \
    .select(
        df_silver["code"],
        df_silver["product_name"],
        df_dim_brand["brand_sk"],
        df_dim_category["category_sk"].alias("primary_category_sk"),
        df_silver["countries_multi"]
    ) \
    .dropDuplicates(["code"]) \
    .withColumn("product_sk", F.monotonically_increasing_id()) \
    .withColumn("effective_from", F.to_date(F.lit(today_str))) \
    .withColumn("effective_to", F.lit(None).cast("date")) \
    .withColumn("is_current", F.lit(True)) \
    .select("product_sk", "code", "product_name", "brand_sk", "primary_category_sk",
            "countries_multi", "effective_from", "effective_to", "is_current")

print(f"dim_product: {df_dim_product.count()} produits")

### 4.7 bridge_product_category (optionnel)
Table de pont pour la relation N-N produit ↔ categorie.

In [ ]:
# CELLULE 19 - bridge_product_category
df_bridge = df_silver \
    .select(F.col("code"), F.explode(F.split(F.col("categories_tags"), ",")).alias("cat_raw")) \
    .withColumn("category_code", F.trim(F.col("cat_raw"))) \
    .filter(F.col("category_code").isNotNull() & (F.col("category_code") != "")) \
    .join(df_dim_product.select("product_sk", "code"), on="code", how="inner") \
    .join(df_dim_category.select("category_sk", "category_code"), on="category_code", how="inner") \
    .select("product_sk", "category_sk") \
    .dropDuplicates()

print(f"bridge_product_category: {df_bridge.count()} relations")

### 4.8 fact_nutrition_snapshot
Table de faits contenant les mesures nutritionnelles et scores.

In [ ]:
# CELLULE 20 - fact_nutrition_snapshot

# Preparer les donnees avec date
df_fact_prep = df_silver.withColumn("snapshot_date",
    F.when(F.col("last_modified_datetime").isNotNull(),
           F.to_date(F.col("last_modified_datetime")))
    .otherwise(F.current_date()))

# Joindre avec dimensions
df_fact = df_fact_prep \
    .join(df_dim_product.select("product_sk", "code"), on="code", how="inner") \
    .join(df_dim_time.select(F.col("time_sk"), F.col("date").alias("snapshot_date")), 
          on="snapshot_date", how="left") \
    .withColumn("time_sk", F.coalesce(F.col("time_sk"), F.lit(1))) \
    .withColumn("fact_id", F.monotonically_increasing_id()) \
    .select(
        "fact_id", "product_sk", "time_sk",
        "energy_kcal_100g", "fat_100g", "saturated_fat_100g", "sugars_100g",
        "salt_100g", "proteins_100g", "fiber_100g", "sodium_100g",
        "nutriscore_grade", "nova_group", "ecoscore_grade",
        "completeness_score", "quality_issues_json"
    )

print(f"fact_nutrition_snapshot: {df_fact.count()} faits")

---
## 5. Chargement PostgreSQL

Chargement des dimensions et faits dans PostgreSQL via JDBC.  
**Strategie** : TRUNCATE CASCADE + INSERT (idempotent).

In [ ]:
# CELLULE 21 - Chargement PostgreSQL
print("=" * 50)
print("CHARGEMENT POSTGRESQL")
print("=" * 50)

counts = {}

print("\nDimensions:")
counts["dim_time"] = load_to_pg(df_dim_time, "dim_time")
counts["dim_brand"] = load_to_pg(df_dim_brand, "dim_brand")
counts["dim_country"] = load_to_pg(df_dim_country, "dim_country")
counts["dim_category"] = load_to_pg(df_dim_category, "dim_category")
counts["dim_nutri"] = load_to_pg(df_dim_nutri, "dim_nutri")

print("\nProduits:")
counts["dim_product"] = load_to_pg(df_dim_product, "dim_product")

print("\nBridge:")
counts["bridge"] = load_to_pg(df_bridge, "bridge_product_category")

print("\nFaits:")
counts["fact"] = load_to_pg(df_fact, "fact_nutrition_snapshot")

print("\n" + "=" * 50)
print("CHARGEMENT TERMINE")
print("=" * 50)

---
## 6. Metriques de qualite

Publication d'un rapport JSON contenant :
- Statistiques du pipeline (avant/apres)
- Comptages par table
- Indicateurs de qualite

In [ ]:
# CELLULE 22 - Metriques JSON
metrics = {
    "run_timestamp": datetime.now().isoformat(),
    "source_file": INPUT_PATH,
    "pipeline": {
        "bronze_count": bronze_count,
        "silver_count": silver_count,
        "filtered_count": bronze_count - silver_count,
        "retention_rate": round(silver_count / bronze_count * 100, 2)
    },
    "dimensions": {
        "dim_time": counts.get("dim_time", 0),
        "dim_brand": counts.get("dim_brand", 0),
        "dim_country": counts.get("dim_country", 0),
        "dim_category": counts.get("dim_category", 0),
        "dim_nutri": counts.get("dim_nutri", 0),
        "dim_product": counts.get("dim_product", 0),
        "bridge_product_category": counts.get("bridge", 0)
    },
    "facts": {
        "fact_nutrition_snapshot": counts.get("fact", 0)
    },
    "quality": {
        "avg_completeness": round(float(avg_completeness or 0), 4),
        "pct_nutriscore": round(df_fact.filter(F.col("nutriscore_grade").isNotNull()).count() / df_fact.count() * 100, 2),
        "pct_nova": round(df_fact.filter(F.col("nova_group").isNotNull()).count() / df_fact.count() * 100, 2),
        "pct_ecoscore": round(df_fact.filter(F.col("ecoscore_grade").isNotNull()).count() / df_fact.count() * 100, 2)
    }
}

print("=" * 50)
print("METRIQUES DE QUALITE (JSON)")
print("=" * 50)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

---
## Resume Final

In [ ]:
# CELLULE 23 - Resume
print("\n" + "=" * 60)
print("ETL OPENFOODFACTS - RESUME")
print("=" * 60)

print(f"\n📊 PIPELINE:")
print(f"   Bronze: {bronze_count} → Silver: {silver_count} ({silver_count/bronze_count*100:.1f}%)")

print(f"\n📦 DIMENSIONS:")
print(f"   dim_time:     {counts.get('dim_time', 0):>6} dates")
print(f"   dim_brand:    {counts.get('dim_brand', 0):>6} marques")
print(f"   dim_country:  {counts.get('dim_country', 0):>6} pays")
print(f"   dim_category: {counts.get('dim_category', 0):>6} categories")
print(f"   dim_nutri:    {counts.get('dim_nutri', 0):>6} scores")
print(f"   dim_product:  {counts.get('dim_product', 0):>6} produits")
print(f"   bridge:       {counts.get('bridge', 0):>6} relations")

print(f"\n📈 FAITS:")
print(f"   fact_nutrition: {counts.get('fact', 0)} lignes")

print(f"\n✅ QUALITE:")
print(f"   Completude moyenne: {avg_completeness:.2%}")
print(f"   Nutri-Score renseigne: {metrics['quality']['pct_nutriscore']:.1f}%")
print(f"   NOVA renseigne: {metrics['quality']['pct_nova']:.1f}%")

In [ ]:
# CELLULE 24 - Fermer Spark
spark.stop()
print("Spark ferme - ETL termine")